# Analyse games with KataGo on Google Colab

Use the **control panel** below. You can choose settings, start the server, copy its connection address and stop it without editing code.

During setup, the panel shows the current step, elapsed time and three recent log lines. If a step is quiet, it shows the time since the last log. Full logs remain below.

## Open the panel

1. Choose **Runtime > Change runtime type > T4 GPU**. GPU availability depends on your plan and usage. Start with the latest runtime version.
2. Click **▶ on the Open control panel block directly below this guide**. Wait for the panel to appear in that block. This is the only code block you need to run. It prepares the controls without logging in, starting a server, deploying an app or running Stop.
3. Keep Medium transformer and HumanSL selected. Click **Start server** and allow Google Drive access when prompted. The first start can take several minutes.
4. Copy the complete address in **SWHub URL** into **Tools > KataGo > Settings > Remote**, then select **Save**. Use the status dot or **Retry connection** if needed.
5. When finished, click **Stop server** in this panel.

Keep the connection address private. Keep Colab connected while using the server. Temporary tunnel addresses can change after restarting.

## Runtime and saved files

Colab does not guarantee a GPU or an uninterrupted session. Free managed runtimes restrict remote-control and primarily web-service use. Use an eligible paid runtime with a positive compute-unit balance when required. [Read Colab’s rules](https://research.google.com/colaboratory/faq.html).

Completed downloads use `MyDrive/SWHub/KataGoAnalysis` when Drive cache is checked. Automatic GPU tuning results are cached too. Keep HumanSL enabled for Career, Subjects and Game Focus. KataGo runs from runtime-local `/content`. Passwords and incomplete downloads stay out of the Drive cache.

All code blocks are collapsed by default. Expand them only if you want to inspect the implementation. If controls do not appear, reconnect and run the top Open control panel block again. Trust the notebook in local Jupyter. After changing settings, stop an active Colab server first, then start again. Modal changes take effect after Deploy.


In [ ]:
# @title ▶ Open control panel
# Generated by tool/katago_notebooks/sync_controls.py.
# Implementation sources remain readable in the collapsed cells below.
%pip install -q "ipywidgets>=7.7.1,<9"

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

"""Shared controls embedded in all four standalone KataGo notebooks.

Run sync_controls.py after editing. No repository checkout is needed by users.
"""

import ast
import builtins
from collections import deque
import html
import json
import re
import subprocess
import threading
import time
from pathlib import Path

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import display


# Registering a cell saves its source without running setup or starting a GPU.
SWHUB_ACTIONS = {}


def swhub_action(line, cell):
    SWHUB_ACTIONS[line.strip()] = cell


get_ipython().register_magic_function(swhub_action, "cell", "swhub_action")


def swhub_configure_source(source, settings):
    """Replace only top-level setting assignments, preserving embedded servers."""
    tree = ast.parse(source)
    lines = source.splitlines(keepends=True)
    for node in reversed(tree.body):
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if isinstance(target, ast.Name) and target.id in settings:
            lines[node.lineno - 1:node.end_lineno] = [
                f"{target.id} = {settings[target.id]!r}\n"
            ]
    return "".join(lines)


def swhub_run_action(name, settings=None, on_output=None):
    source = SWHUB_ACTIONS[name]
    # Existing writefile cells remain readable, self-contained service sources.
    lines = source.splitlines(keepends=True)
    writefile = next((i for i, line in enumerate(lines)
                      if line.startswith("%%writefile ")), None)
    if writefile is not None:
        path = Path(lines[writefile].split(maxsplit=1)[1].strip())
        source = swhub_configure_source("".join(lines[writefile + 1:]), settings or {})
        compile(source, str(path), "exec")
        path.write_text(source, encoding="utf-8")
        print("Prepared", path.name)
        if on_output is not None:
            on_output("Prepared " + path.name)
        return
    source = swhub_configure_source(source, settings or {})
    # Observe Python action logs without redirecting the kernel's stdout or
    # moving Drive authorization and IPython work off the calling thread.
    had_print = "print" in globals()
    original_print = globals().get("print", builtins.print)

    def report(*args, **kwargs):
        original_print(*args, **kwargs)
        if on_output is not None and kwargs.get("file") is None:
            separator = kwargs.get("sep", " ")
            on_output((" " if separator is None else separator).join(str(arg) for arg in args))

    globals()["print"] = report
    try:
        exec(compile(source, f"KataGo action: {name}", "exec"), globals())
    finally:
        if had_print:
            globals()["print"] = original_print
        else:
            globals().pop("print", None)


class SWHubKataGoPanel:
    def __init__(self, kind):
        self.kind = kind
        self.modal = kind.startswith("modal")
        self.contribution = kind.endswith("contribution")
        self.busy = False
        self.target = None
        self.activity_stop = threading.Event()
        self.activity_lock = threading.RLock()
        self.recent_activity = deque(maxlen=3)
        self.activity_secrets = ()
        self.buttons = {}
        self.fields = {}
        self.status = widgets.HTML(value="Ready. Choose your settings, then start below.")
        self.activity = widgets.HTML(layout=widgets.Layout(max_height="120px", overflow="auto"))
        self.endpoint = widgets.Text(description="SWHub URL", placeholder="Address appears after starting",
                                     layout=widgets.Layout(width="100%"))
        self.connection = widgets.HTML()
        self.output = widgets.Output(layout=widgets.Layout(
            max_height="180px", overflow="auto", border="1px solid #888"))
        self.progress = widgets.IntProgress(min=0, max=1, description="Progress")
        children = [widgets.HTML(value="<h2>SWHub · KataGo " +
                    ("Contribution" if self.contribution else "Analysis") + "</h2>")]
        if self.modal:
            children += [widgets.HTML(value="Modal supplies the T4 GPU. GPU use requires a valid "
                         "payment method. Closing this notebook does not stop the deployed app."),
                         self.button("login", "1. Login", self.login),
                         self.field("MODAL_PROFILE", widgets.Dropdown, "Workspace", options=[]),
                         self.field("MODAL_ENVIRONMENT", widgets.Text, "Environment", value="main"),
                         self.button("profiles", "Refresh workspaces", self.refresh_profiles),
                         self.button("select", "2. Select workspace", self.select_workspace)]
            self.fields["MODAL_PROFILE"].observe(self.invalidate_target, names="value")
            self.fields["MODAL_ENVIRONMENT"].observe(self.invalidate_target, names="value")
        if not self.contribution:
            children += [self.field("MAIN_MODEL_PRESET", widgets.Dropdown, "Model", options=[
                ("Medium transformer (recommended)", "transformer_medium"),
                ("Small transformer", "transformer_small"),
                ("Large transformer", "transformer_large"),
                ("Convolutional b18", "colab_fast_b18"),
                ("Convolutional b28", "latest_b28"),
                ("Convolutional b40 Zhizi", "strongest_b40_zhizi"),
                ("Custom model", "custom")]),
                self.field("ENABLE_HUMAN_MODEL", widgets.Checkbox, "HumanSL", value=True),
                widgets.HTML(value="Keep HumanSL enabled for Career, Subjects and Game Focus.")]
            custom = [self.field(key, widgets.Text, label) for key, label in (
                ("MODEL_URL", "Model URL"), ("MODEL_FILENAME", "Filename"),
                ("MODEL_SHA256", "SHA-256"))]
            advanced = widgets.Accordion(children=[widgets.VBox(custom)], selected_index=None)
            advanced.set_title(0, "Custom model settings (only for Custom model)")
            children.append(advanced)
        else:
            children.append(widgets.HTML(value="The training server chooses the model. Enter your "
                            "katagotraining.org account in SWHub. Finish games & stop in SWHub "
                            "before stopping this server. Sessions last up to four hours."))
        if self.kind == "colab_analysis":
            children += [self.field("TUNNEL_PROVIDER", widgets.Dropdown, "Connection", options=[
                ("Automatic (recommended)", "auto"), ("Cloudflare", "cloudflare"),
                ("Pinggy", "pinggy"), ("localhost.run", "localhostrun"),
                ("Local only", "none")]),
                self.field("USE_GOOGLE_DRIVE_CACHE", widgets.Checkbox, "Drive cache", value=True)]
        if self.kind == "modal_contribution":
            children += [self.field("ENABLE_CONTRIBUTOR_LOGIN", widgets.Checkbox,
                                    "Controller login", value=False),
                         self.field("CONTRIBUTOR_LOGIN_USERNAME", widgets.Text, "Username"),
                         self.field("CONTRIBUTOR_LOGIN_PASSWORD", widgets.Password, "Password"),
                         widgets.HTML(value="Optional extra server login, separate from your training account. "
                                      "Enter it again for each deployment. Keep the server URL private.")]
        children += [self.button("start", "3. Deploy" if self.modal else "Start server",
                                 self.start, "success"), self.progress, self.status, self.activity,
                     widgets.HTML(value="<b>Connect SWHub</b>. Copy the full address below into " +
                                  ("Tools → KataGo Contribution → Settings." if self.contribution else
                                   "Tools → KataGo → Settings → Remote, then select Save.")),
                     self.endpoint, self.connection,
                     self.button("logs", "Show recent logs", self.logs),
                     self.button("stop", "Stop Modal app" if self.modal else "Stop server",
                                 self.stop, "danger")]
        children += [widgets.HTML(value="<b>Full setup and troubleshooting logs</b>. "
                                  "Wait for an action to finish before clicking another. "
                                  "Use the notebook's Interrupt execution control if setup stalls."), self.output]
        self.view = widgets.VBox(children, layout=widgets.Layout(max_width="850px"))
        self.update_buttons()

    def field(self, key, widget_type, description, **kwargs):
        field = widget_type(description=description, style={"description_width": "120px"},
                            layout=widgets.Layout(width="100%"), **kwargs)
        self.fields[key] = field
        return field

    def button(self, key, description, action, style=""):
        button = widgets.Button(description=description, button_style=style,
                                layout=widgets.Layout(width="230px"))
        button.on_click(lambda _: self.perform(description, action))
        self.buttons[key] = button
        return button

    def update_buttons(self):
        for button in self.buttons.values():
            button.disabled = self.busy
        for field in self.fields.values():
            field.disabled = self.busy
        if self.modal and not self.busy:
            for key in ("start", "logs", "stop"):
                self.buttons[key].disabled = self.target is None
            self.buttons["select"].disabled = not self.fields["MODAL_PROFILE"].value

    def invalidate_target(self, change=None):
        self.target = None
        globals()["selected_modal_target"] = None
        self.endpoint.value = ""
        self.connection.value = ""
        self.status.value = "Select workspace to verify the new deployment target."
        self.update_buttons()

    def perform(self, label, action):
        if self.busy:
            return
        self.busy = True
        self.update_buttons()
        self.status.value = html.escape(label) + "…"
        self.progress.bar_style = "info"
        self.progress.max = 1
        self.progress.value = 0
        self.started_at = self.last_activity_at = time.monotonic()
        self.last_render_at = 0
        self.recent_activity.clear()
        secret = self.fields.get("CONTRIBUTOR_LOGIN_PASSWORD")
        self.activity_secrets = (secret.value,) if secret is not None and secret.value else ()
        self.activity_stop = threading.Event()
        self.render_activity()
        self.activity_thread = threading.Thread(target=self.watch_activity, daemon=True)
        self.activity_thread.start()
        with self.output:
            print("\n" + label)
            try:
                action()
            except (Exception, KeyboardInterrupt) as exc:
                self.progress.bar_style = "danger"
                self.status.value = "Action interrupted." if isinstance(exc, KeyboardInterrupt) else (
                    "Action failed. Read the details below, correct the problem and retry.")
                # Passwords are never printed in tracebacks or saved to notebook source.
                message = str(exc)
                for secret in self.activity_secrets:
                    message = message.replace(secret, "[redacted]")
                print(type(exc).__name__ + ": " + message)
            else:
                self.progress.bar_style = "success"
                self.progress.value = self.progress.max
                self.status.value = ("Server address ready. Connect SWHub below." if action == self.start
                                     else html.escape(label) + " complete.")
            finally:
                self.activity_stop.set()
                self.activity_thread.join(timeout=2)
                self.render_activity(finished=True)
                if action == self.start:
                    self.clear_controller_password()
                self.activity_secrets = ()
                self.busy = False
                self.update_buttons()

    def run_action(self, name, settings=None):
        swhub_run_action(name, settings, on_output=self.record_activity)

    def record_activity(self, message):
        with self.activity_lock:
            if not self.busy or self.activity_stop.is_set():
                return
            # Keep the preview bounded, strip terminal escapes, and escape HTML.
            message = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", message)
            for secret in self.activity_secrets:
                message = message.replace(secret, "[redacted]")
            for line in message.replace("\r", "\n").splitlines()[-3:]:
                line = " ".join(line.split())
                if not line:
                    continue
                self.last_activity_at = time.monotonic()
                elapsed = int(self.last_activity_at - self.started_at)
                self.recent_activity.append((elapsed, line[:180]))
                stage = re.match(r"\[(\d+)/(\d+)\]", line)
                if stage:
                    self.status.value = html.escape(line[:180])
                    self.progress.max = max(1, int(stage[2]))
                    self.progress.value = min(self.progress.max, max(0, int(stage[1]) - 1))
            if time.monotonic() - self.last_render_at >= 0.2:
                self.render_activity()

    def render_activity(self, finished=False):
        with self.activity_lock:
            now = time.monotonic()
            self.last_render_at = now
            elapsed = int(now - self.started_at)
            quiet = int(now - self.last_activity_at)
            timing = f"Elapsed {elapsed // 60}:{elapsed % 60:02d}"
            if not finished:
                timing += (f" · No new log for {quiet}s. Waiting for this step to finish."
                           if quiet >= 10 else " · Working")
            lines = [f"<div><small>{timing}</small></div>"]
            for stamp, line in self.recent_activity:
                lines.append(f"<div>{stamp // 60}:{stamp % 60:02d} · {html.escape(line)}</div>")
            self.activity.value = '<div style="overflow-wrap:anywhere">' + "".join(lines) + "</div>"

    def watch_activity(self):
        while not self.activity_stop.wait(1):
            self.render_activity()

    def refresh_profiles(self):
        self.invalidate_target()
        # Reject token overrides before interpreting CLI output as profiles.
        import os
        if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
            raise RuntimeError("Token environment variables override workspace profiles. "
                               "Remove them or use a clean runtime, then Login again.")
        result = subprocess.run(["modal", "profile", "list", "--json"],
                                check=True, text=True, capture_output=True, timeout=60)
        profiles = json.loads(result.stdout)
        names = [row["name"] for row in profiles]
        self.fields["MODAL_PROFILE"].options = names
        print("Choose a workspace and select it." if names else "No saved workspaces. Login first.")

    def login(self):
        self.invalidate_target()
        self.run_action("login")
        self.refresh_profiles()

    def select_workspace(self):
        self.invalidate_target()
        self.run_action("workspace", {key: self.fields[key].value for key in (
            "MODAL_PROFILE", "MODAL_ENVIRONMENT")})
        self.target = globals()["selected_modal_target"]

    def settings(self):
        values = {key: field.value for key, field in self.fields.items()}
        if not self.contribution:
            values["HUMAN_MODEL_PRESET"] = (
                "official_human_sl_v0" if values["ENABLE_HUMAN_MODEL"] else "disabled")
            if values["MAIN_MODEL_PRESET"] == "custom":
                from urllib.parse import urlparse
                import re
                if urlparse(values["MODEL_URL"]).scheme != "https":
                    raise ValueError("Enter an HTTPS URL for the custom model.")
                filename = values["MODEL_FILENAME"]
                if not filename or filename in (".", "..") or "/" in filename or "\\" in filename:
                    raise ValueError("Enter a model filename without folders.")
                if re.fullmatch(r"[0-9a-fA-F]{64}", values["MODEL_SHA256"]) is None:
                    raise ValueError("Enter the custom model's 64-character SHA-256.")
            else:
                for key in ("MODEL_URL", "MODEL_FILENAME", "MODEL_SHA256"):
                    values.pop(key)
        if self.modal:
            values["KEEP_STREAMING_AFTER_READY"] = False
        return values

    def start(self):
        if not self.modal:
            for name in (("service", "tunnel") if self.contribution else ("ws_proc", "tunnel_proc")):
                process = globals().get(name)
                if process is not None and process.poll() is None:
                    raise RuntimeError("Server is already running. Stop server before starting again.")
        self.endpoint.value = ""
        self.connection.value = ""
        for key in ("final_wss_url", "final_status_url", "websocket_url", "base", "ready"):
            globals().pop(key, None)
        settings = self.settings()
        if self.modal and (self.target is None or self.target != globals().get("selected_modal_target")):
            self.invalidate_target()
            raise RuntimeError("Select workspace again before deploying.")
        if self.kind == "modal_contribution":
            if settings["ENABLE_CONTRIBUTOR_LOGIN"] and not all(settings[key].strip() for key in (
                    "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")):
                raise ValueError("Enter both controller login fields or disable Controller login.")
            globals().update({key: settings[key] for key in (
                "ENABLE_CONTRIBUTOR_LOGIN", "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")})
        steps = (["write_app", "deploy"] if self.modal else
                 ["prepare", "drive", "write_app", "start"] if self.contribution else ["start"])
        self.progress.max = len(steps)
        self.progress.value = 0
        try:
            for index, step in enumerate(steps):
                stage = {"prepare": "Checking GPU and installing dependencies",
                         "drive": "Connecting Google Drive and restoring cached files",
                         "write_app": "Preparing the server",
                         "deploy": "Deploying to Modal, waiting for startup",
                         "start": "Preparing KataGo and opening the connection"}[step]
                self.status.value = f"Step {index + 1}/{len(steps)}: {stage}…"
                self.record_activity(stage)
                self.run_action(step, settings)
                if self.progress.max == len(steps):
                    self.progress.value = index + 1
            if self.kind == "colab_analysis":
                url = globals().get("final_wss_url", "")
                status_url = globals().get("final_status_url", "")
            elif self.kind == "modal_contribution":
                url = globals().get("websocket_url", "")
                status_url = ""
            else:
                base = globals().get("base", "")
                url = base.replace("https://", "wss://").replace("http://", "ws://") + (
                    "/contribute" if self.contribution else "/katago") if base else ""
                status_url = base + ("/" if self.contribution else "/status") if base else ""
            if not url:
                raise RuntimeError("No connection URL found. Check the deployment logs below.")
            self.endpoint.value = url
            if status_url:
                self.connection.value = '<a target="_blank" rel="noopener" href="' + html.escape(
                    status_url, quote=True) + '">Open server status</a>'
            if self.kind == "modal_analysis" and not globals().get("ready"):
                raise RuntimeError("Deployment finished, but KataGo readiness was not confirmed. "
                                   "Check recent logs or open server status before connecting.")
            print("Connection address ready. Keep it private.")
        finally:
            self.clear_controller_password()

    def clear_controller_password(self):
        if self.kind == "modal_contribution":
            self.fields["CONTRIBUTOR_LOGIN_PASSWORD"].value = ""
            # The existing deployment cell keeps its subprocess environment in globals.
            for key in ("CONTRIBUTOR_LOGIN_PASSWORD", "password"):
                globals().pop(key, None)
            environment = globals().get("env", {})
            environment.pop("SWHUB_CONTRIBUTOR_PASSWORD", None)
            environment.pop("SWHUB_CONTRIBUTOR_USERNAME", None)

    def logs(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before viewing logs.")
            app = "Katago-Contribute" if self.contribution else "KataGo-Analyse"
            # Recent logs return promptly, the deploy action already streams startup.
            subprocess.run(modal_command("app", "logs", app), check=True, timeout=30)
        else:
            paths = (["/content/colab_katago_contribute.log", "/content/colab_cloudflared.log"]
                     if self.contribution else ["/content/katago_ws_server.log", "/content/cloudflared.log"])
            for filename in paths:
                path = Path(filename)
                if path.exists():
                    print(path.name + "\n" + path.read_text(errors="replace")[-12000:])

    def stop(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before stopping.")
        self.run_action("stop")
        self.endpoint.value = ""
        self.connection.value = ""


def show_swhub_panel(kind):
    previous = globals().get("swhub_panel")
    if previous is not None:
        previous.activity_stop.set()
        for button in previous.buttons.values():
            button.disabled = True
        previous.view.close()
    panel = SWHubKataGoPanel(kind)
    globals()["swhub_panel"] = panel
    display(panel.view)
    return panel


# Load the bundled actions without executing setup or starting a server.
import base64 as _swhub_base64
import zlib as _swhub_zlib
_swhub_action_bundle = '...'
SWHUB_ACTIONS.update(json.loads(_swhub_zlib.decompress(
    _swhub_base64.b64decode(_swhub_action_bundle))))
swhub_panel = show_swhub_panel('colab_analysis')


## Advanced implementation (optional)

These code blocks are collapsed by default. The top Open control panel block already includes everything the panel needs, so you do not need to run the blocks below. Expand them to inspect the implementation.


In [ ]:
%%swhub_action start
# @title Register start (runs only from the panel)
# @title Start KataGo Remote Engine - edit user settings below


# CUDA-only runtime. Select a Colab T4 GPU (or another available NVIDIA GPU).
# Setup stops with a clear error if Colab did not attach a usable GPU.

# If True, old server/tunnel/KataGo processes are stopped before starting.
STOP_OLD_PROCESSES = True

# Persist completed KataGo/model/cloudflared assets between Colab runtimes.
# KataGo still runs from fast /content storage, incomplete downloads are excluded.
USE_GOOGLE_DRIVE_CACHE = True
GOOGLE_DRIVE_CACHE_PATH = "/content/drive/MyDrive/SWHub/KataGoAnalysis"

# Tunnel provider:
#   "auto"         -> try TUNNEL_FALLBACK_ORDER in order
#   "cloudflare"   -> Cloudflare Quick Tunnel, no account, random trycloudflare URL
#   "pinggy"       -> Pinggy free SSH tunnel, no download, 60-minute free timeout
#   "localhostrun" -> localhost.run free SSH tunnel, no download, changing free domains
#   "none"         -> skip public tunnel, print local ws/http URLs only
TUNNEL_PROVIDER = "auto"
TUNNEL_FALLBACK_ORDER = ["cloudflare", "pinggy", "localhostrun"]

# Tunnel startup timeouts.
CLOUDFLARE_TIMEOUT_SECONDS = 120
PINGGY_TIMEOUT_SECONDS = 90
LOCALHOSTRUN_TIMEOUT_SECONDS = 90

# SSH tunnel options for Pinggy/localhost.run fallbacks.
SSH_SERVER_ALIVE_INTERVAL_SECONDS = 30
SSH_CONNECT_TIMEOUT_SECONDS = 15
PINGGY_DEBUG_PORT = 4300

# If True, re-download KataGo even if the folder already exists.
FORCE_REDOWNLOAD_KATAGO = False

# If True, re-download model even if model.bin.gz already exists.
FORCE_REDOWNLOAD_MODEL = False

# If True, re-download Human SL model even if it already exists.
FORCE_REDOWNLOAD_HUMAN_MODEL = False

# Backward-compatible aliases for older pasted copies of the settings block.
FORCE_REDOWLOAD_KATAGO = FORCE_REDOWNLOAD_KATAGO
FORCE_REDOWLOAD_MODEL = FORCE_REDOWNLOAD_MODEL
FORCE_REDOWLOAD_HUMAN_MODEL = FORCE_REDOWNLOAD_HUMAN_MODEL

# KataGo 1.18.2 release URLs. Transformer models require 1.17 or newer.
KATAGO_RELEASE_CACHE_KEY = "v1_18_2"
GPU_CUDA_CUDNN9_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.18.2/katago-v1.18.2-cuda12.1-cudnn9.8.0-linux-x64.zip"
GPU_CUDA_CUDNN9_KATAGO_SHA256 = "3e30b486b7bc38287eeead350aa832ff8ca50d1f0c0a83b5409cabf8a3bc9c5d"
GPU_CUDA_CUDNN8_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.18.2/katago-v1.18.2-cuda12.1-cudnn8.9.7-linux-x64.zip"
GPU_CUDA_CUDNN8_KATAGO_SHA256 = "16c69f42291fe8c6d196d722d92299a5b04de852611d6c022a0dd9e0e83b5688"

# Main analysis model. The medium transformer is the recommended default.
MAIN_MODEL_PRESET = "transformer_medium"
MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_FILENAME = "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_SHA256 = "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995"

# Human SL model support.
# Keep this True if you use SWHub human-like/rank/policy features.
# KataGo needs this extra model when SWHub sends overrideSettings.humanSLProfile.
ENABLE_HUMAN_MODEL = True
HUMAN_MODEL_PRESET = "official_human_sl_v0"
HUMAN_MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_FILENAME = "b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_SHA256 = "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5"

# Local self-test profile for Human SL. SWHub can still choose other profiles.
# Valid examples include rank_20k...rank_9d, preaz_20k...preaz_9d, proyear_1800...proyear_2023.
HUMAN_MODEL_TEST_PROFILE = "rank_4k"

# GPU config defaults. SWHub can override visits per query.
GPU_MAX_VISITS = 500
GPU_SEARCH_THREADS = 4
GPU_ANALYSIS_THREADS = 4
GPU_NN_CACHE_POWER = 21
# KataGo v1.18.2 benchmarks the backend-derived batch size against half
# the search threads and checks whether two CUDA NN servers are faster.
# Results are cached per GPU/model. Set either override to an integer only
# when you deliberately want to bypass the measured value.
AUTO_TUNE_GPU_SETTINGS = True
GPU_TUNING_VISITS = 200
GPU_TUNING_POSITIONS = 3
GPU_NN_MAX_BATCH_SIZE = None
GPU_NN_SERVER_THREADS = None
GPU_CUDA_DEVICE = 0

# Local server port inside Colab.
SERVER_PORT = 8000

# WebSocket route used by SWHub.
KATAGO_WS_PATH = "/katago"

# Large KataGo JSON output buffer.
SUBPROCESS_BUFFER_MB = 50

# How long to wait for KataGo startup.
# First GPU run with Human SL can tune two models and take several minutes.
KATAGO_STARTUP_TIMEOUT_SECONDS = 900

# Print latest server log every N seconds while waiting for startup.
STARTUP_LOG_INTERVAL_SECONDS = 15
STARTUP_LOG_TAIL_CHARS = 2200

# Diagnostic log memory.
ENGINE_LOG_LIMIT = 1200

# ============================================================
# Script starts here. Usually do not edit below this line.
# ============================================================

import os
import sys
import tempfile
import hashlib
import re
import json
import time
import signal
import shutil
import subprocess
import urllib.request
from pathlib import Path

RUNTIME_BASE = Path("/content")
BASE = RUNTIME_BASE / "swhub-katago-analysis"
BASE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_DIR = Path(GOOGLE_DRIVE_CACHE_PATH)
CLOUDFLARED_PATH = BASE / "cloudflared"
CLOUDFLARED_VERSION = "2026.10.7"
CLOUDFLARED_URL = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
CLOUDFLARED_SHA256 = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
CLOUDFLARE_LOG_PATH = RUNTIME_BASE / "cloudflared.log"
PINGGY_LOG_PATH = RUNTIME_BASE / "pinggy_tunnel.log"
LOCALHOSTRUN_LOG_PATH = RUNTIME_BASE / "localhostrun_tunnel.log"
WS_SERVER_PATH = RUNTIME_BASE / "katago_ws_server.py"
WS_LOG_PATH = RUNTIME_BASE / "katago_ws_server.log"

MAIN_MODEL_PRESETS = {
    "transformer_small": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c384h6nbttflrs.bin.gz",
        "filename": "b10c384h6nbttflrs.bin.gz",
        "sha256": "0ba27eced5180b3e3d0b898b280c541112989765e789d1eb6cd0d31b2b2c1229",
        "note": "Recommended compact transformer, stronger per visit than b18 and usually as fast or faster.",
    },
    "transformer_medium": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "filename": "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "sha256": "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995",
        "note": "Medium transformer, stronger per visit than b28 and usually as fast or faster.",
    },
    "transformer_large": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-tf3-b11c768-s12002M-d6304M.bin.gz",
        "filename": "kata1-tf3-b11c768-s12002M-d6304M.bin.gz",
        "sha256": "4a6312e80faadee7b7dd28689a2e87a1efb4640c10132f16290da7a17b4c6d9e",
        "note": "Largest transformer, stronger than b40 Zhizi and intended for capable runtimes.",
    },
    "colab_fast_b18": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "filename": "kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "sha256": "9d7a6afed8ff5b74894727e156f04f0cd36060a24824892008fbb6e0cba51f1d",
        "note": "Fast lightweight b18 model, good for free Colab.",
    },
    "latest_b28": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "filename": "kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "sha256": "c5bca453d7b08ea8df6546439325d4dd681e77d975e1da3f7593d771147b73bc",
        "note": "Latest official kata1 b28 network as of 2026-06-07, stronger but slower.",
    },
    "strongest_b40_zhizi": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "filename": "kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "sha256": "15fb3baf85cdb6578e6c19b65e6201ca906cec3ba5fee19039d05221d57eb0e8",
        "note": "Strongest confidently-rated network on katagotraining, heavy for Colab.",
    },
}

HUMAN_MODEL_PRESETS = {
    "official_human_sl_v0": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz",
        "filename": "b18c384nbt-humanv0.bin.gz",
        "sha256": "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5",
        "note": "Official Human SL model for humanSLProfile/rank-like analysis.",
    },
}


def apply_model_presets():
    global MODEL_URL, MODEL_FILENAME, MODEL_SHA256
    global HUMAN_MODEL_URL, HUMAN_MODEL_FILENAME, HUMAN_MODEL_SHA256
    global ENABLE_HUMAN_MODEL

    main_preset = MAIN_MODEL_PRESET.lower().strip()
    if main_preset != "custom":
        if main_preset not in MAIN_MODEL_PRESETS:
            choices = sorted([*MAIN_MODEL_PRESETS.keys(), "custom"])
            raise ValueError(f"MAIN_MODEL_PRESET must be one of {choices}")
        data = MAIN_MODEL_PRESETS[main_preset]
        MODEL_URL = data["url"]
        MODEL_FILENAME = data["filename"]
        MODEL_SHA256 = data["sha256"]

    human_preset = HUMAN_MODEL_PRESET.lower().strip()
    if human_preset == "disabled":
        ENABLE_HUMAN_MODEL = False
    elif human_preset != "custom":
        if human_preset not in HUMAN_MODEL_PRESETS:
            choices = sorted([*HUMAN_MODEL_PRESETS.keys(), "custom", "disabled"])
            raise ValueError(f"HUMAN_MODEL_PRESET must be one of {choices}")
        data = HUMAN_MODEL_PRESETS[human_preset]
        HUMAN_MODEL_URL = data["url"]
        HUMAN_MODEL_FILENAME = data["filename"]
        HUMAN_MODEL_SHA256 = data["sha256"]


def run(cmd, check=True):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout


def command_output(cmd):
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return result.returncode, result.stdout


def detected_gpu_name():
    code, out = command_output("nvidia-smi --query-gpu=name --format=csv,noheader")
    if code != 0 or not out.strip():
        return None
    return out.strip().splitlines()[0].strip()


# Keep this self-contained helper identical in both Colab notebooks.
# CUDA 13-only images do not provide the CUDA 12 SONAMEs used by KataGo.
CUDA_RUNTIME_DIR = Path('/content/swhub-katago-cuda12-12.8-cudnn9.8')
CUDA_REQUIREMENTS = '''\
nvidia-cuda-runtime-cu12==12.8.90 --hash=sha256:adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90
nvidia-cublas-cu12==12.8.4.1 --hash=sha256:8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142
nvidia-cudnn-cu12==9.8.0.87 --hash=sha256:d6b02cd0e3e24aa31d0193a8c39fec239354360d7d81055edddb69f35d53a4c8
nvidia-cuda-nvrtc-cu12==12.8.93 --hash=sha256:a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994
nvidia-nvjitlink-cu12==12.8.93 --hash=sha256:81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88
'''


def _configure_cuda_library_path(runtime_root=None):
    # A new child process must see these paths, changing the notebook's
    # environment does not change the loader paths of its running Python.
    runtime_root = CUDA_RUNTIME_DIR if runtime_root is None else runtime_root
    directories = [str(path.resolve()) for path in sorted(runtime_root.glob('nvidia/*/lib'))]
    directories.extend(part for part in os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep) if part)
    for entry in sys.path:
        if not entry:
            continue
        root = Path(entry)
        for component in ('cublas', 'cuda_runtime', 'cudnn', 'cuda_nvrtc', 'nvjitlink'):
            for directory in (root / 'nvidia' / component / 'lib', root / component / 'lib'):
                if directory.is_dir():
                    directories.append(str(directory.resolve()))
        for directory in (root / 'nvidia' / 'cu12' / 'lib', root / 'torch' / 'lib'):
            if directory.is_dir():
                directories.append(str(directory.resolve()))
    for pattern in ('cuda-12*/lib64', 'cuda-12*/targets/x86_64-linux/lib', 'cuda/lib64'):
        for directory in sorted(Path('/usr/local').glob(pattern)):
            if directory.is_dir():
                directories.append(str(directory.resolve()))
    if directories:
        os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(dict.fromkeys(directories))


def _probe_cuda_runtime(allow_cudnn8=False, minimum_cuda=12080):
    # Probe the standalone loader in a fresh process, never Torch's privately
    # preloaded libraries. Include cuDNN's lazily loaded component libraries.
    probe = r'''
import ctypes, json, sys
cuda = ctypes.CDLL('libcudart.so.12')
version = ctypes.c_int()
if cuda.cudaRuntimeGetVersion(ctypes.byref(version)) != 0 or not int(sys.argv[2]) <= version.value < 13000:
    raise RuntimeError('A compatible CUDA 12 runtime is required')
for name in ('libcublas.so.12', 'libcublasLt.so.12', 'libnvrtc.so.12', 'libnvJitLink.so.12'):
    ctypes.CDLL(name)
errors = []
for major in ([9, 8] if sys.argv[1] == '1' else [9]):
    try:
        cudnn = ctypes.CDLL(f'libcudnn.so.{major}')
        cudnn.cudnnGetVersion.restype = ctypes.c_size_t
        value = cudnn.cudnnGetVersion()
        parts = (value // 10000, value % 10000 // 100, value % 100) if value >= 10000 else (value // 1000, value % 1000 // 100, value % 100)
        if parts[0] != major or parts < ((9, 8, 0) if major == 9 else (8, 9, 7)):
            raise RuntimeError(f'Incompatible cuDNN version {parts}')
        components = ('ops', 'cnn', 'adv', 'graph', 'heuristic', 'engines_precompiled', 'engines_runtime_compiled') if major == 9 else ('ops_infer', 'cnn_infer', 'adv_infer')
        for component in components:
            ctypes.CDLL(f'libcudnn_{component}.so.{major}')
        print(json.dumps(parts))
        break
    except (OSError, RuntimeError) as error:
        errors.append(str(error))
else:
    raise RuntimeError('; '.join(errors))
'''
    result = subprocess.run(
        [sys.executable, '-c', probe, '1' if allow_cudnn8 else '0', str(minimum_cuda)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=60,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return tuple(json.loads(result.stdout))


def _ensure_cuda_runtime(allow_cudnn8=False, minimum_cuda=12080):
    _configure_cuda_library_path()
    try:
        return _probe_cuda_runtime(allow_cudnn8, minimum_cuda)
    except RuntimeError as error:
        print('Standalone CUDA 12 library check failed:', error, flush=True)
    print('Installing compatible NVIDIA CUDA 12 libraries for KataGo (first setup may take several minutes)...', flush=True)
    # Isolate the pinned libraries from Colab's Torch/TensorFlow packages and
    # Drive model caches. Publish only after installation and loader validation.
    CUDA_RUNTIME_DIR.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix='swhub-cuda-install-', dir=CUDA_RUNTIME_DIR.parent))
    previous_path = os.environ.get('LD_LIBRARY_PATH')
    try:
        requirements = staging / 'requirements.txt'
        requirements.write_text(CUDA_REQUIREMENTS)
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
             '--no-input', '--no-deps', '--only-binary=:all:', '--require-hashes',
             '--target', str(staging), '-r', str(requirements)],
            check=True, timeout=1800,
        )
        _configure_cuda_library_path(staging)
        _probe_cuda_runtime(False, minimum_cuda)
        if CUDA_RUNTIME_DIR.exists():
            shutil.rmtree(CUDA_RUNTIME_DIR)
        staging.rename(CUDA_RUNTIME_DIR)
    except Exception as error:
        raise RuntimeError(f'Could not prepare KataGo CUDA 12 libraries: {error}') from error
    finally:
        if staging.exists():
            shutil.rmtree(staging)
        if previous_path is None:
            os.environ.pop('LD_LIBRARY_PATH', None)
        else:
            os.environ['LD_LIBRARY_PATH'] = previous_path
    _configure_cuda_library_path()
    version = _probe_cuda_runtime(False, minimum_cuda)
    print('KataGo CUDA 12 libraries ready; cuDNN ' + '.'.join(map(str, version)), flush=True)
    return version


def choose_cuda_runtime():
    gpu_name = detected_gpu_name()
    if not gpu_name:
        raise RuntimeError(
            "This notebook requires a Colab NVIDIA GPU runtime. "
            "Choose Runtime -> Change runtime type -> T4 GPU and rerun."
        )
    cudnn_version = _ensure_cuda_runtime(allow_cudnn8=True, minimum_cuda=12010)
    use_cudnn9 = (9, 8, 0) <= cudnn_version < (10, 0, 0)
    use_cudnn8 = (8, 9, 7) <= cudnn_version < (9, 0, 0)
    if not use_cudnn9 and not use_cudnn8:
        detected = ".".join(str(part) for part in cudnn_version)
        raise RuntimeError(
            f"Detected cuDNN {detected}, KataGo v1.18 requires "
            "cuDNN >= 9.8.0 or the compatibility cuDNN 8.9.7 runtime."
        )
    cudnn_label = ".".join(str(part) for part in cudnn_version)
    return {
        "mode": "gpu",
        "backend": "cuda",
        "mode_name": f"{gpu_name} CUDA (cuDNN {cudnn_label})",
        "gpu_name": gpu_name,
        "katago_cache_dir": BASE / f"katago_{KATAGO_RELEASE_CACHE_KEY}_cuda_cudnn{9 if use_cudnn9 else 8}",
        "katago_dir": RUNTIME_BASE / f"katago_{KATAGO_RELEASE_CACHE_KEY}_cuda_cudnn{9 if use_cudnn9 else 8}",
        "katago_url": GPU_CUDA_CUDNN9_KATAGO_URL if use_cudnn9 else GPU_CUDA_CUDNN8_KATAGO_URL,
        "katago_sha256": GPU_CUDA_CUDNN9_KATAGO_SHA256 if use_cudnn9 else GPU_CUDA_CUDNN8_KATAGO_SHA256,
    }


CONFIG_DATA = choose_cuda_runtime()
apply_model_presets()
KATAGO_CACHE_DIR = CONFIG_DATA["katago_cache_dir"]
KATAGO_DIR = CONFIG_DATA["katago_dir"]
MODEL_DIR = BASE / "models"


def stop_old_processes():
    print("\n[1/12] Stopping old processes...")
    current_pid = os.getpid()

    try:
        out = subprocess.check_output(["ps", "-eo", "pid=,args="], text=True)
    except Exception as exc:
        print("Could not list processes:", exc)
        return

    kill_keywords = [
        "katago_ws_server.py",
        "cloudflared tunnel",
        "katago analysis",
        "free.pinggy.io",
        "localhost.run",
    ]

    for raw_line in out.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split(None, 1)
        if len(parts) < 2:
            continue
        if not parts[0].isdigit():
            continue

        pid = int(parts[0])
        cmd = parts[1]

        if pid == current_pid:
            continue

        if any(keyword in cmd for keyword in kill_keywords):
            print("Stopping:", pid, cmd)
            try:
                os.kill(pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            except PermissionError:
                print("No permission to stop:", pid)

    time.sleep(2)


def install_dependencies(include_analysis=False):
    print("Installing the packages needed by this notebook...")
    print("Python:", sys.version.split()[0])
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        # Ubuntu 24.04 uses the t64 name, Ubuntu 22.04 uses libzip4.
        zip_probe = subprocess.run(
            ["apt-cache", "show", "libzip4t64"],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            check=False,
        )
        zip_package = "libzip4t64" if zip_probe.returncode == 0 and zip_probe.stdout.strip() else "libzip4"
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "ca-certificates", "curl",
             "wget", "p7zip-full", "rsync", "unzip", "libgomp1", "libomp-dev",
             "openssh-client", zip_package],
            check=True,
        )
        packages = ["fastapi==0.162.2", "uvicorn==0.54.0", "websockets==17.4"]
        if include_analysis:
            packages.append("nest_asyncio==1.6.0")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *packages],
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Package installation failed. Read the command error above and retry Start server. "
            "If the problem started after a Colab runtime update, see the notebook's Troubleshooting section."
        ) from exc


def restore_persistent_cache():
    if not USE_GOOGLE_DRIVE_CACHE:
        print("\nPersistent Google Drive cache disabled, using this runtime only.")
        return
    from google.colab import drive
    print("\n[2/12] Connecting Google Drive and restoring cached files...")
    drive.mount("/content/drive")
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    BASE.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "rsync", "-a", "--delete", "--delete-excluded",
            "--exclude=*.tmp", "--exclude=*.tmp.*",
            f"{DRIVE_CACHE_DIR}/", f"{BASE}/",
        ],
        check=True,
    )
    print("Restored persistent analysis assets from:", DRIVE_CACHE_DIR)


def sync_persistent_cache():
    if not USE_GOOGLE_DRIVE_CACHE:
        return
    if not DRIVE_CACHE_DIR.parent.is_dir():
        raise RuntimeError("Google Drive is not mounted, cannot save the persistent cache.")
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    BASE.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "rsync", "-a", "--delete-delay", "--delete-excluded",
            "--exclude=*.tmp", "--exclude=*.tmp.*",
            f"{BASE}/", f"{DRIVE_CACHE_DIR}/",
        ],
        check=True,
    )
    print("Saved persistent analysis assets to:", DRIVE_CACHE_DIR)


def print_system_info():
    print("\n[3/12] System info...")
    print("Selected mode:", CONFIG_DATA["mode_name"])
    print("Main model preset:", MAIN_MODEL_PRESET)
    if MAIN_MODEL_PRESET.lower().strip() in MAIN_MODEL_PRESETS:
        print("Main model note:", MAIN_MODEL_PRESETS[MAIN_MODEL_PRESET.lower().strip()]["note"])
    print("Main model URL:", MODEL_URL)
    print("Human model preset:", HUMAN_MODEL_PRESET)
    if ENABLE_HUMAN_MODEL and HUMAN_MODEL_PRESET.lower().strip() in HUMAN_MODEL_PRESETS:
        print("Human model note:", HUMAN_MODEL_PRESETS[HUMAN_MODEL_PRESET.lower().strip()]["note"])
    print("Human model enabled:", ENABLE_HUMAN_MODEL)
    print("Runtime cache folder:", BASE)
    print("Persistent Drive cache:", DRIVE_CACHE_DIR if USE_GOOGLE_DRIVE_CACHE else "disabled")
    print("KataGo cache folder:", KATAGO_CACHE_DIR)
    print("KataGo folder:", KATAGO_DIR)
    print("Model folder:", MODEL_DIR)
    print("Assigned NVIDIA GPU:", CONFIG_DATA["gpu_name"])
    run("nvidia-smi", check=False)
    run("lscpu | head -80", check=False)


def stage_katago_runtime():
    katago_bin = KATAGO_DIR / "katago"

    if KATAGO_CACHE_DIR == KATAGO_DIR:
        run(f"chmod +x {katago_bin}", check=False)
        return

    if KATAGO_DIR.exists():
        shutil.rmtree(KATAGO_DIR)

    print(f"Staging KataGo executable files into runnable runtime folder: {KATAGO_DIR}")
    shutil.copytree(
        KATAGO_CACHE_DIR,
        KATAGO_DIR,
        ignore=shutil.ignore_patterns("*.zip", "models", "logs"),
    )
    run(f"chmod +x {katago_bin}", check=True)


def validated_katago_executable():
    katago_bin = KATAGO_DIR / "katago"
    version_pattern = r"(?:^|[^0-9])1\.18\.2(?:[^0-9]|$)"
    direct = subprocess.run(
        [str(katago_bin), "version"], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
    )
    print(direct.stdout, flush=True)
    if direct.returncode == 0:
        if not re.search(version_pattern, direct.stdout):
            raise RuntimeError(f"Expected KataGo 1.18.2, got:\n{direct.stdout}")
        return katago_bin

    print("KataGo did not run directly. Trying AppImage extraction fallback...")
    extracted_bin = KATAGO_DIR / "squashfs-root" / "AppRun"
    if not extracted_bin.exists():
        extracted = subprocess.run(
            [str(katago_bin), "--appimage-extract"], cwd=KATAGO_DIR,
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            check=False, timeout=120,
        )
        if extracted.returncode != 0:
            raise RuntimeError(
                "KataGo AppImage extraction failed after the direct executable failed:\n"
                + direct.stdout + "\n" + extracted.stdout
            )
    if not extracted_bin.exists():
        raise RuntimeError(f"KataGo AppImage extraction did not create {extracted_bin}")
    # Drive-restored caches can lose execute bits on the wrapped binary.
    (extracted_bin.parent / "usr" / "bin" / "katago").chmod(0o755)
    extracted_bin.chmod(0o755)
    fallback = subprocess.run(
        [str(extracted_bin), "version"], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
    )
    print(fallback.stdout, flush=True)
    if fallback.returncode != 0:
        raise RuntimeError(f"Extracted CUDA KataGo failed to run:\n{fallback.stdout}")
    if not re.search(version_pattern, fallback.stdout):
        raise RuntimeError(f"Expected KataGo 1.18.2, got:\n{fallback.stdout}")
    return extracted_bin


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_download(path, expected_sha256=""):
    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Download produced an empty file: {path}")
    if expected_sha256:
        actual = sha256_file(path)
        if actual.lower() != expected_sha256.lower():
            raise RuntimeError(f"SHA-256 mismatch for {path.name}: expected {expected_sha256}, got {actual}")


def download_file(url, path, expected_sha256="", force=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not force:
        try:
            verify_download(path, expected_sha256)
            print(f"Already exists and verified: {path}")
            return
        except Exception as exc:
            print(f"Cached file failed verification, downloading again: {exc}")
    tmp = path.with_name(path.name + ".tmp")
    tmp.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "SWHub-KataGo-Colab/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(tmp, "wb") as target:
            shutil.copyfileobj(response, target)
    except Exception as exc:
        print(f"Python download failed ({exc!r}), trying wget...")
        tmp.unlink(missing_ok=True)
        subprocess.run(["wget", "--tries=3", "--timeout=60", "-O", str(tmp), url], check=True)
    try:
        verify_download(tmp, expected_sha256)
        os.replace(tmp, path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def download_katago():
    print("\n[4/12] Downloading KataGo...")
    KATAGO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(KATAGO_CACHE_DIR)

    cache_bin = KATAGO_CACHE_DIR / "katago"
    archive = KATAGO_CACHE_DIR / "katago.zip"

    def download_and_extract():
        download_file(
            CONFIG_DATA["katago_url"], archive,
            CONFIG_DATA["katago_sha256"], force=True,
        )
        run("unzip -o katago.zip", check=True)

    if FORCE_REDOWNLOAD_KATAGO or not cache_bin.exists():
        download_and_extract()
    else:
        print("KataGo already exists in cache. Validating it before reuse.")

    for attempt in range(2):
        stage_katago_runtime()
        try:
            return validated_katago_executable()
        except Exception as exc:
            if attempt == 0 and not FORCE_REDOWNLOAD_KATAGO:
                print("Cached KataGo runtime failed validation, repairing it from the verified release archive.")
                print(exc)
                shutil.rmtree(KATAGO_CACHE_DIR)
                KATAGO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
                os.chdir(KATAGO_CACHE_DIR)
                download_and_extract()
                continue
            raise RuntimeError(
                f"CUDA KataGo failed to run on {CONFIG_DATA['gpu_name']}: {exc}"
            ) from exc


def download_model():
    print("\n[5/12] Downloading model files...")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model_path = MODEL_DIR / MODEL_FILENAME
    download_file(MODEL_URL, model_path, MODEL_SHA256, force=FORCE_REDOWNLOAD_MODEL)

    run(f'ls -lh "{model_path}"', check=True)

    if ENABLE_HUMAN_MODEL:
        human_model_path = MODEL_DIR / HUMAN_MODEL_FILENAME
        print("Checking Human SL model...")
        download_file(HUMAN_MODEL_URL, human_model_path, HUMAN_MODEL_SHA256, force=FORCE_REDOWNLOAD_HUMAN_MODEL)

        run(f'ls -lh "{human_model_path}"', check=True)
    else:
        print("Human SL model disabled by ENABLE_HUMAN_MODEL = False")


def _gpu_tuning_cache_path():
    identity = "|".join([
        KATAGO_RELEASE_CACHE_KEY,
        CONFIG_DATA["mode_name"],
        os.environ.get("LD_LIBRARY_PATH", ""),
        MODEL_SHA256 or MODEL_FILENAME,
        str(GPU_SEARCH_THREADS),
        str(GPU_ANALYSIS_THREADS),
    ])
    key = hashlib.sha256(identity.encode()).hexdigest()[:24]
    return BASE / "tuning" / f"cuda_{key}.json"


def _write_analysis_config_values(server_threads=None, max_batch_size=None, include_benchmark_threads=False):
    server_lines = []
    if server_threads is not None:
        server_lines.append(f"numNNServerThreadsPerModel = {server_threads}")
    device_count = server_threads or 1
    server_lines.extend(
        f"cudaDeviceToUseThread{index} = {GPU_CUDA_DEVICE}"
        for index in range(device_count)
    )
    if max_batch_size is None:
        max_batch_size = max(1, GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS)
    server_lines.append(f"nnMaxBatchSize = {max_batch_size}")
    backend_config = "\n".join(server_lines)

    benchmark_threads = f"numSearchThreads = {GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS}\n" if include_benchmark_threads else ""
    analysis_threads = "" if include_benchmark_threads else (
        f"numSearchThreadsPerAnalysisThread = {GPU_SEARCH_THREADS}\n"
        f"numAnalysisThreads = {GPU_ANALYSIS_THREADS}\n"
    )
    cfg = f"""logDir = {KATAGO_DIR}/logs

maxVisits = {GPU_MAX_VISITS}

{benchmark_threads}{analysis_threads}

nnCacheSizePowerOfTwo = {GPU_NN_CACHE_POWER}
{backend_config}
"""
    config_path = KATAGO_DIR / "analysis.cfg"
    config_path.write_text(cfg)
    return config_path, cfg


def measured_gpu_settings(katago_exec):
    cache_path = _gpu_tuning_cache_path()
    if AUTO_TUNE_GPU_SETTINGS and cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text())
            if cached.get("schema") == 2:
                print("Using cached KataGo GPU tuning:", cache_path)
                return cached.get("server_threads"), cached.get("max_batch_size")
        except Exception as exc:
            print("Ignoring invalid GPU tuning cache:", exc)

    if not AUTO_TUNE_GPU_SETTINGS:
        return None, None

    config_path, _ = _write_analysis_config_values(include_benchmark_threads=True)
    command = [
        str(katago_exec), "benchmark",
        "-model", str(MODEL_DIR / MODEL_FILENAME),
        "-config", str(config_path),
        "-threads", str(GPU_SEARCH_THREADS * GPU_ANALYSIS_THREADS),
        "-visits", str(GPU_TUNING_VISITS),
        "-numpositions", str(GPU_TUNING_POSITIONS),
    ]
    print("\n[6/12] Validating and tuning this GPU/model with KataGo v1.18.2...")
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=900,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError("KataGo GPU/model benchmark failed, the selected runtime is not usable.")
    server_threads = 2 if "ADDITIONAL RECOMMENDATION: 2 NN server threads" in result.stdout else 1
    batch_match = re.search(
        r"ADDITIONAL RECOMMENDATION: a smaller batch size.*?nnMaxBatchSize = (\d+)",
        result.stdout, flags=re.DOTALL,
    )
    max_batch_size = int(batch_match.group(1)) if batch_match else None
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps({
        "schema": 2,
        "server_threads": server_threads,
        "max_batch_size": max_batch_size,
    }, indent=2))
    return server_threads, max_batch_size


def write_analysis_config(katago_exec):
    print("\n[6/12] Writing analysis.cfg...")

    measured_servers, measured_batch = measured_gpu_settings(katago_exec)
    server_threads = GPU_NN_SERVER_THREADS if GPU_NN_SERVER_THREADS is not None else measured_servers
    max_batch_size = GPU_NN_MAX_BATCH_SIZE if GPU_NN_MAX_BATCH_SIZE is not None else measured_batch
    config_path, cfg = _write_analysis_config_values(server_threads, max_batch_size)
    print("Selected config mode:", CONFIG_DATA["mode_name"])
    print("Wrote:", config_path)
    print(cfg)


def write_ws_server(katago_exec):
    print("\n[7/12] Writing WebSocket server with status and log diagnostics...")

    server_template = r'''
import asyncio
import json
import time
import uuid
from collections import deque
from typing import Dict, Tuple

import nest_asyncio
import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import PlainTextResponse

MODE_NAME = __MODE_NAME__
KATAGO_BIN = __KATAGO_BIN__
MODEL = __MODEL__
HUMAN_MODEL = __HUMAN_MODEL__
CONFIG = __CONFIG__
SERVER_PORT = __SERVER_PORT__
SUBPROCESS_LIMIT = __SUBPROCESS_LIMIT__
LOG_LIMIT = __LOG_LIMIT__

app = FastAPI()
katago_proc = None
# Internal IDs are unique per submission, client IDs belong to one connection.
pending: Dict[str, Tuple[WebSocket, asyncio.Lock, str]] = {}
client_queries: Dict[WebSocket, Dict[str, str]] = {}
pending_expected_turns: Dict[str, set] = {}
pending_completed_turns: Dict[str, set] = {}
katago_write_lock = asyncio.Lock()
LOGS = deque(maxlen=LOG_LIMIT)
STALE_RESULT_IDS = set()
STALE_RESULT_ORDER = deque()

STATE = {
    "mode": MODE_NAME,
    "started_at": time.time(),
    "katago_started": False,
    "katago_ready": False,
    "websocket_clients_total": 0,
    "websocket_clients_active": 0,
    "queries_received": 0,
    "results_sent": 0,
    "errors": 0,
    "warnings": 0,
    "last_query_id": None,
    "last_result_id": None,
    "last_error": None,
    "last_best_move": None,
    "last_visits": None,
    "last_winrate": None,
    "last_score_lead": None,
    "human_model_enabled": bool(HUMAN_MODEL),
    "pending_queries": 0,
    "stale_results_dropped": 0,
}


def now_time():
    return time.strftime("%H:%M:%S")


def now_ts():
    return time.time()


async def emit(event_type, message, extra=None):
    if event_type == "error":
        STATE["errors"] += 1
    if event_type == "warning":
        STATE["warnings"] += 1

    STATE["pending_queries"] = len(pending)

    item = {
        "ts": now_ts(),
        "time": now_time(),
        "type": event_type,
        "message": str(message),
        "state": dict(STATE),
    }
    if extra:
        item.update(extra)

    LOGS.append(item)

    print(f"[{item['time']}] {event_type}: {message}", flush=True)


def first_stale_result(query_id):
    if not query_id or query_id in STALE_RESULT_IDS:
        return False
    STALE_RESULT_IDS.add(query_id)
    STALE_RESULT_ORDER.append(query_id)
    while len(STALE_RESULT_ORDER) > 500:
        STALE_RESULT_IDS.discard(STALE_RESULT_ORDER.popleft())
    return True


async def write_to_katago(payload):
    if katago_proc is None or katago_proc.stdin is None:
        raise RuntimeError("KataGo process is not running.")
    if katago_proc.returncode is not None:
        raise RuntimeError(f"KataGo process already exited with code {katago_proc.returncode}.")

    async with katago_write_lock:
        katago_proc.stdin.write((json.dumps(payload) + "\n").encode("utf-8"))
        await katago_proc.stdin.drain()


def finish_pending_query(query_id):
    target = pending.pop(query_id, None)
    if target:
        websocket, _, client_id = target
        query_ids = client_queries.get(websocket, {})
        if query_ids.get(client_id) == query_id:
            query_ids.pop(client_id, None)
    pending_expected_turns.pop(query_id, None)
    pending_completed_turns.pop(query_id, None)
    STATE["pending_queries"] = len(pending)


async def terminate_katago_query(query_id, reason, retain_until_final=False, turn_numbers=None):
    if not retain_until_final:
        finish_pending_query(query_id)
    terminate = {"id": f"terminate-{query_id}-{uuid.uuid4()}", "action": "terminate", "terminateId": query_id}
    if turn_numbers is not None:
        terminate["turnNumbers"] = turn_numbers
    try:
        await write_to_katago(terminate)
        await emit("query", f"Terminated query id={query_id}: {reason}", {"query_id": query_id})
    except Exception as exc:
        STATE["last_error"] = repr(exc)
        await emit("error", f"Could not terminate query id={query_id}: {repr(exc)}")


async def terminate_pending_query(query_id, reason):
    if query_id not in pending:
        return
    await terminate_katago_query(query_id, reason)


async def start_katago():
    global katago_proc
    await emit("system", f"Starting KataGo mode={MODE_NAME}")

    katago_args = [KATAGO_BIN, "analysis", "-model", MODEL, "-config", CONFIG]

    if HUMAN_MODEL:
        katago_args.extend(["-human-model", HUMAN_MODEL])
        await emit("system", f"Human SL model enabled: {HUMAN_MODEL}")
    else:
        await emit("warning", "Human SL model disabled. SWHub humanSLProfile queries will fail.")

    await emit("system", "Launch command: " + " ".join(katago_args))

    katago_proc = await asyncio.create_subprocess_exec(
        *katago_args,
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        limit=SUBPROCESS_LIMIT,
    )

    STATE["katago_started"] = True
    asyncio.create_task(read_katago_stdout())
    asyncio.create_task(read_katago_stderr())
    await emit("system", "KataGo process started.")


async def read_katago_stdout():
    while True:
        try:
            line = await katago_proc.stdout.readline()
        except Exception as exc:
            STATE["last_error"] = repr(exc)
            await emit("error", f"KataGo stdout read error: {repr(exc)}")
            break

        if not line:
            await emit("system", "KataGo stdout closed.")
            break

        text = line.decode("utf-8", errors="ignore").strip()
        if not text:
            continue

        try:
            data = json.loads(text)
            query_id = data.get("id")
            terminate_id = data.get("terminateId")
            STATE["last_result_id"] = query_id

            if terminate_id:
                await emit("result", f"KataGo acknowledged termination id={terminate_id}", {"query_id": terminate_id})
                continue

            if query_id is None and (data.get("error") or data.get("warning") or data.get("noResults")):
                detail = data.get("error") or data.get("warning") or "KataGo returned no results"
                level = "error" if data.get("error") or data.get("noResults") else "warning"
                await emit(level, f"Unscoped KataGo {level}: {detail}")
                delivered = set()
                for websocket, send_lock, _ in list(pending.values()):
                    if id(websocket) in delivered:
                        continue
                    delivered.add(id(websocket))
                    try:
                        async with send_lock:
                            await websocket.send_text(json.dumps(data))
                    except Exception as exc:
                        await emit("warning", f"Could not forward unscoped KataGo error: {exc!r}")
                continue

            target = pending.get(query_id)
            if not target:
                STATE["stale_results_dropped"] += 1
                if first_stale_result(query_id):
                    await emit("stale", f"Dropped stale result id={query_id}, client already cancelled or replaced this query")
                continue

            websocket, send_lock, client_id = target
            async with send_lock:
                await websocket.send_text(json.dumps(dict(data, id=client_id)))

            STATE["results_sent"] += 1
            move_infos = data.get("moveInfos", [])
            best_move = move_infos[0].get("move") if move_infos else None
            visits = move_infos[0].get("visits") if move_infos else None
            winrate = move_infos[0].get("winrate") if move_infos else None
            score_lead = move_infos[0].get("scoreLead") if move_infos else None

            STATE["last_best_move"] = best_move
            STATE["last_visits"] = visits
            STATE["last_winrate"] = winrate
            STATE["last_score_lead"] = score_lead

            await emit(
                "result",
                f"Sent result id={query_id}, best={best_move}, visits={visits}, winrate={winrate}, scoreLead={score_lead}",
                {"query_id": query_id, "best_move": best_move, "visits": visits, "winrate": winrate, "score_lead": score_lead},
            )

            if data.get("error"):
                finish_pending_query(query_id)
            elif data.get("noResults") or data.get("isDuringSearch") is False:
                expected = pending_expected_turns.get(query_id, set())
                if not expected:
                    finish_pending_query(query_id)
                else:
                    turn_number = data.get("turnNumber")
                    if turn_number is not None:
                        pending_completed_turns.setdefault(query_id, set()).add(int(turn_number))
                    if expected.issubset(pending_completed_turns.get(query_id, set())):
                        finish_pending_query(query_id)

        except Exception as exc:
            STATE["last_error"] = repr(exc)
            await emit("error", f"Error forwarding KataGo output: {repr(exc)}")


async def read_katago_stderr():
    while True:
        line = await katago_proc.stderr.readline()
        if not line:
            break

        msg = line.decode("utf-8", errors="ignore").rstrip()
        lower = msg.lower()

        if "started, ready to begin handling requests" in lower:
            STATE["katago_ready"] = True
            await emit("katago", "KataGo ready.")
        elif "using cuda backend" in lower or "loaded model" in lower:
            await emit("katago", msg)
        elif "performing autotuning" in lower or "done tuning" in lower or "saved results" in lower:
            await emit("tuning", msg)
        elif "tuning " in msg or "ErrorProp" in msg or "Calls/sec" in msg:
            await emit("tuning", msg)
        elif "warning" in lower:
            await emit("warning", msg)
        elif "error" in lower or "failed" in lower:
            STATE["last_error"] = msg
            await emit("error", msg)
        else:
            print("KataGo:", msg, flush=True)


@app.on_event("startup")
async def startup_event():
    await start_katago()


@app.get("/")
async def home():
    return {
        "status": "ok",
        "message": "KataGo WebSocket server running",
        "mode": MODE_NAME,
        "endpoints": {"status": "/status", "logs": "/logs", "katago_websocket": "/katago"},
    }


@app.get("/status")
async def status():
    STATE["pending_queries"] = len(pending)
    return {
        "ok": True,
        "uptime_seconds": round(time.time() - STATE["started_at"], 2),
        "state": STATE,
        "pending_queries": len(pending),
        "recent_logs": list(LOGS)[-50:],
    }


@app.get("/logs", response_class=PlainTextResponse)
async def logs(type: str = "all", q: str = ""):
    rows = list(LOGS)
    if type and type != "all":
        wanted = {t.strip() for t in type.split(",") if t.strip()}
        rows = [item for item in rows if item.get("type") in wanted]
    if q:
        q_lower = q.lower()
        rows = [item for item in rows if q_lower in item.get("message", "").lower()]
    return "\n".join(f"[{item['time']}] {item['type']}: {item['message']}" for item in rows)


async def handle_katago_websocket(websocket: WebSocket):
    await websocket.accept()
    send_lock = asyncio.Lock()
    query_ids = {}
    client_queries[websocket] = query_ids
    STATE["websocket_clients_total"] += 1
    STATE["websocket_clients_active"] += 1
    await emit("client", f"WebSocket client connected. Active clients: {STATE['websocket_clients_active']}")

    try:
        while True:
            message = await websocket.receive_text()
            try:
                query = json.loads(message)
            except Exception as exc:
                STATE["last_error"] = f"Bad JSON from websocket: {repr(exc)}"
                await emit("error", f"Bad JSON from websocket: {repr(exc)}")
                await emit("error", f"Message preview: {message[:500]}")
                continue

            if query.get("action") == "terminate":
                target_id = query_ids.get(query.get("terminateId"))
                if target_id:
                    await terminate_katago_query(target_id, "client requested termination", retain_until_final=True, turn_numbers=query.get("turnNumbers"))
                async with send_lock:
                    await websocket.send_text(json.dumps(query))
                continue
            if query.get("action") == "terminate_all":
                # A client's stop-all must not cancel another connection's work.
                for target_id in list(query_ids.values()):
                    await terminate_katago_query(target_id, "client requested termination", retain_until_final=True, turn_numbers=query.get("turnNumbers"))
                async with send_lock:
                    await websocket.send_text(json.dumps(query))
                continue
            client_id = query.get("id") or str(uuid.uuid4())
            previous_id = query_ids.get(client_id)
            if previous_id:
                await terminate_pending_query(previous_id, "replaced by newer query with the same id")
            query_id = str(uuid.uuid4())
            query["id"] = query_id
            query_ids[client_id] = query_id
            STATE["queries_received"] += 1
            STATE["last_query_id"] = query_id
            pending[query_id] = (websocket, send_lock, client_id)
            pending_expected_turns[query_id] = {int(turn) for turn in (query.get("analyzeTurns") or [])}
            pending_completed_turns[query_id] = set()
            STATE["pending_queries"] = len(pending)

            move_count = len(query.get("moves", []))
            max_visits = query.get("maxVisits")
            analyze_turns = query.get("analyzeTurns")
            override_settings = query.get("overrideSettings")
            await emit(
                "query",
                f"Received query id={query_id}, moves={move_count}, maxVisits={max_visits}, analyzeTurns={analyze_turns}, overrideSettings={override_settings}",
                {"query_id": query_id, "move_count": move_count, "max_visits": max_visits, "analyze_turns": analyze_turns, "override_settings": override_settings},
            )

            await write_to_katago(query)

    except WebSocketDisconnect:
        await emit("client", "WebSocket client disconnected.")
    except Exception as exc:
        STATE["last_error"] = repr(exc)
        await emit("error", f"WebSocket error: {repr(exc)}")
    finally:
        STATE["websocket_clients_active"] = max(0, STATE["websocket_clients_active"] - 1)
        dead_ids = [qid for qid, item in pending.items() if item[0] is websocket]
        for qid in dead_ids:
            await terminate_katago_query(qid, "websocket disconnected")
        client_queries.pop(websocket, None)
        STATE["pending_queries"] = len(pending)
        await emit("client", f"Client cleanup done. Active clients: {STATE['websocket_clients_active']}")


@app.websocket("/")
async def websocket_root(websocket: WebSocket):
    await handle_katago_websocket(websocket)


@app.websocket("/katago")
async def websocket_katago(websocket: WebSocket):
    await handle_katago_websocket(websocket)


nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=SERVER_PORT, ws="websockets")
'''

    server_code = server_template
    server_code = server_code.replace("__MODE_NAME__", json.dumps(CONFIG_DATA["mode_name"]))
    server_code = server_code.replace("__KATAGO_BIN__", json.dumps(str(katago_exec)))
    server_code = server_code.replace("__MODEL__", json.dumps(str(MODEL_DIR / MODEL_FILENAME)))
    human_model_path = MODEL_DIR / HUMAN_MODEL_FILENAME
    server_code = server_code.replace("__HUMAN_MODEL__", json.dumps(str(human_model_path) if ENABLE_HUMAN_MODEL else ""))
    server_code = server_code.replace("__CONFIG__", json.dumps(str(KATAGO_DIR / "analysis.cfg")))
    server_code = server_code.replace("__SERVER_PORT__", str(SERVER_PORT))
    server_code = server_code.replace("__SUBPROCESS_LIMIT__", str(SUBPROCESS_BUFFER_MB * 1024 * 1024))
    server_code = server_code.replace("__LOG_LIMIT__", str(ENGINE_LOG_LIMIT))

    WS_SERVER_PATH.write_text(server_code)
    print(f"Wrote: {WS_SERVER_PATH}")


def start_ws_server():
    print("\n[8/12] Starting WebSocket server...")

    if WS_LOG_PATH.exists():
        WS_LOG_PATH.unlink()

    log = open(WS_LOG_PATH, "w")
    proc = subprocess.Popen([sys.executable, str(WS_SERVER_PATH)], stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("WebSocket server PID:", proc.pid)
    print("Waiting for KataGo startup...")
    started = False

    for i in range(KATAGO_STARTUP_TIMEOUT_SECONDS):
        time.sleep(1)
        if WS_LOG_PATH.exists():
            text = WS_LOG_PATH.read_text(errors="ignore")
            if "KataGo ready" in text or "Started, ready to begin handling requests" in text:
                started = True
                break
            if "Traceback" in text:
                print("\n--- Server log error ---")
                print(text[-8000:])
                raise RuntimeError("WebSocket server crashed.")
        if i % STARTUP_LOG_INTERVAL_SECONDS == 0:
            print(f"Still waiting... {i}s")
            if WS_LOG_PATH.exists():
                log_tail = WS_LOG_PATH.read_text(errors="ignore")[-STARTUP_LOG_TAIL_CHARS:]
                print(log_tail)

    print("\n--- WebSocket server log tail ---")
    print(WS_LOG_PATH.read_text(errors="ignore")[-5000:])

    if not started:
        raise RuntimeError(f"KataGo did not finish starting. Check {WS_LOG_PATH}")

    return proc


def test_local_http():
    print("\n[9/12] Testing local HTTP endpoints...")
    out = run(f"curl -s http://127.0.0.1:{SERVER_PORT}/", check=True)
    if "KataGo WebSocket server running" not in out:
        raise RuntimeError("Local HTTP test failed.")

    out = run(f"curl -s http://127.0.0.1:{SERVER_PORT}/status", check=True)
    if '"ok":true' not in out.replace(" ", ""):
        raise RuntimeError("Local status test failed.")


def test_local_websocket():
    print("\n[10/12] Testing local WebSocket...")
    visits = 10

    override_settings_line = ""
    if ENABLE_HUMAN_MODEL:
        override_settings_line = f',\n        "overrideSettings": {{"humanSLProfile": "{HUMAN_MODEL_TEST_PROFILE}"}}'

    test_code = f'''
import asyncio
import json
import websockets

async def test():
    uri = "ws://127.0.0.1:{SERVER_PORT}{KATAGO_WS_PATH}"
    query = {{
        "id": "local-ws-test",
        "moves": [["B", "Q16"], ["W", "D4"], ["B", "Q4"], ["W", "D16"]],
        "rules": "chinese",
        "komi": 7.5,
        "boardXSize": 19,
        "boardYSize": 19,
        "analyzeTurns": [4],
        "maxVisits": {visits},
        "includeOwnership": False,
        "includePolicy": False{override_settings_line}
    }}
    async with websockets.connect(uri, max_size={SUBPROCESS_BUFFER_MB} * 1024 * 1024) as ws:
        await ws.send(json.dumps(query))
        result = await ws.recv()
        print(result[:1000])

asyncio.run(test())
'''

    test_path = RUNTIME_BASE / "test_local_ws.py"
    test_path.write_text(test_code)
    out = run(f"python {test_path}", check=True)

    if '"moveInfos"' not in out and '"rootInfo"' not in out:
        raise RuntimeError("Local WebSocket test did not return KataGo analysis.")


def download_cloudflared():
    print("\n[11/12] Downloading cloudflared...")
    os.chdir(RUNTIME_BASE)

    download_file(CLOUDFLARED_URL, CLOUDFLARED_PATH, CLOUDFLARED_SHA256)
    run(f"chmod +x {CLOUDFLARED_PATH}", check=True)
    cloudflared_version = run(f"{CLOUDFLARED_PATH} --version", check=True)
    if CLOUDFLARED_VERSION not in cloudflared_version:
        raise RuntimeError(f"Expected cloudflared {CLOUDFLARED_VERSION}, got: {cloudflared_version}")


def make_public_urls(base_url):
    base_url = base_url.rstrip("/")
    return {
        "base": base_url,
        "wss": base_url.replace("https://", "wss://").replace("http://", "ws://") + KATAGO_WS_PATH,
        "status": base_url + "/status",
        "logs": base_url + "/logs",
    }


def print_tunnel_success(provider_name, proc, urls):
    print("\n" + "=" * 90)
    print("COPY THIS INTO SWHub -> KATAGO SETTINGS -> REMOTE -> WEBSOCKET URL")
    print()
    print(urls["wss"])
    print()
    print("=" * 90)

    print("\nTunnel provider:", provider_name)
    print("\nSTATUS:")
    print(urls["status"])

    print("\nOther useful links:")
    print("Status:", urls["status"])
    print("Logs:  ", urls["logs"])
    print("Filtered logs examples:")
    print("Errors:", urls["logs"] + "?type=error")
    print("Tuning:", urls["logs"] + "?type=tuning")
    print("Search:", urls["logs"] + "?q=humanSLProfile")

    print("\nSWHub steps:")
    print("1. Open KataGo settings in SWHub.")
    print("2. Select the Remote tab.")
    print("3. Paste the WebSocket URL printed above.")
    print("4. Click Save.")
    print("5. Click the status dot to reconnect if needed.")
    print("6. Use /status and /logs to inspect the remote engine if needed.")

    return proc, urls["wss"], urls["status"]


def start_cloudflare_tunnel():
    print("\nStarting Cloudflare Quick Tunnel...")

    if CLOUDFLARE_LOG_PATH.exists():
        CLOUDFLARE_LOG_PATH.unlink()

    log = open(CLOUDFLARE_LOG_PATH, "w")
    proc = subprocess.Popen([str(CLOUDFLARED_PATH), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"], stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("cloudflared PID:", proc.pid)
    print("Waiting for trycloudflare URL...")
    url = None

    for i in range(CLOUDFLARE_TIMEOUT_SECONDS):
        time.sleep(1)
        if CLOUDFLARE_LOG_PATH.exists():
            text = CLOUDFLARE_LOG_PATH.read_text(errors="ignore")
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", text)
            if match:
                url = match.group(0)
                break
        if i % 10 == 0:
            print(f"Still waiting for tunnel URL... {i}s")

    print("\n--- Cloudflare log tail ---")
    print(CLOUDFLARE_LOG_PATH.read_text(errors="ignore")[-5000:])

    if not url:
        raise RuntimeError(f"Could not find Cloudflare URL. Check {CLOUDFLARE_LOG_PATH}")

    return print_tunnel_success("cloudflare", proc, make_public_urls(url))


def ssh_common_options():
    return [
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", f"ServerAliveInterval={SSH_SERVER_ALIVE_INTERVAL_SECONDS}",
        "-o", f"ConnectTimeout={SSH_CONNECT_TIMEOUT_SECONDS}",
        "-o", "ExitOnForwardFailure=yes",
    ]


def read_log_tail(path, chars=5000):
    if not path.exists():
        return ""
    return path.read_text(errors="ignore")[-chars:]


def start_pinggy_tunnel():
    print("\nStarting Pinggy free SSH tunnel...")

    if PINGGY_LOG_PATH.exists():
        PINGGY_LOG_PATH.unlink()

    cmd = [
        "ssh",
        *ssh_common_options(),
        "-p", "443",
        "-L", f"{PINGGY_DEBUG_PORT}:localhost:{PINGGY_DEBUG_PORT}",
        "-R", f"0:127.0.0.1:{SERVER_PORT}",
        "free.pinggy.io",
    ]
    log = open(PINGGY_LOG_PATH, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("Pinggy SSH PID:", proc.pid)
    print("Waiting for Pinggy URL...")
    url = None

    for i in range(PINGGY_TIMEOUT_SECONDS):
        time.sleep(1)
        text = read_log_tail(PINGGY_LOG_PATH, 12000)
        match = re.search(r"https://[a-zA-Z0-9\-.]+pinggy(?:-free)?\.link", text)
        if match:
            url = match.group(0)
            break

        code, out = command_output(f"curl -s http://127.0.0.1:{PINGGY_DEBUG_PORT}/urls")
        if code == 0:
            match = re.search(r"https://[a-zA-Z0-9\-.]+pinggy(?:-free)?\.link", out)
            if match:
                url = match.group(0)
                break

        if proc.poll() is not None:
            raise RuntimeError(f"Pinggy tunnel exited early. Check {PINGGY_LOG_PATH}")
        if i % 10 == 0:
            print(f"Still waiting for Pinggy URL... {i}s")

    print("\n--- Pinggy log tail ---")
    print(read_log_tail(PINGGY_LOG_PATH))

    if not url:
        raise RuntimeError(f"Could not find Pinggy URL. Check {PINGGY_LOG_PATH}")

    return print_tunnel_success("pinggy", proc, make_public_urls(url))


def start_localhostrun_tunnel():
    print("\nStarting localhost.run free SSH tunnel...")

    if LOCALHOSTRUN_LOG_PATH.exists():
        LOCALHOSTRUN_LOG_PATH.unlink()

    cmd = [
        "ssh",
        *ssh_common_options(),
        "-R", f"80:127.0.0.1:{SERVER_PORT}",
        "nokey@localhost.run",
    ]
    log = open(LOCALHOSTRUN_LOG_PATH, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("localhost.run SSH PID:", proc.pid)
    print("Waiting for localhost.run URL...")
    url = None

    for i in range(LOCALHOSTRUN_TIMEOUT_SECONDS):
        time.sleep(1)
        text = read_log_tail(LOCALHOSTRUN_LOG_PATH, 12000)
        matches = re.findall(r"https://[a-zA-Z0-9\-._~:/?#\[\]@!$&'()*+,;=%]+", text)
        for candidate in matches:
            if "localhost.run" in candidate or "lhr.life" in candidate or "lhr.rocks" in candidate:
                url = candidate.rstrip(".,)")
                break
        if url:
            break

        if proc.poll() is not None:
            raise RuntimeError(f"localhost.run tunnel exited early. Check {LOCALHOSTRUN_LOG_PATH}")
        if i % 10 == 0:
            print(f"Still waiting for localhost.run URL... {i}s")

    print("\n--- localhost.run log tail ---")
    print(read_log_tail(LOCALHOSTRUN_LOG_PATH))

    if not url:
        raise RuntimeError(f"Could not find localhost.run URL. Check {LOCALHOSTRUN_LOG_PATH}")

    return print_tunnel_success("localhostrun", proc, make_public_urls(url))


def selected_tunnel_providers():
    provider = TUNNEL_PROVIDER.lower().strip()
    allowed = {"auto", "cloudflare", "pinggy", "localhostrun", "none"}
    if provider not in allowed:
        raise ValueError(f"TUNNEL_PROVIDER must be one of {sorted(allowed)}")
    if provider == "none":
        return []
    if provider != "auto":
        return [provider]

    providers = []
    for item in TUNNEL_FALLBACK_ORDER:
        item = str(item).lower().strip()
        if item not in {"cloudflare", "pinggy", "localhostrun"}:
            raise ValueError(f"Unknown tunnel provider in TUNNEL_FALLBACK_ORDER: {item}")
        if item not in providers:
            providers.append(item)
    return providers


def start_public_tunnel():
    print("\n[12/12] Starting public tunnel...")
    providers = selected_tunnel_providers()
    if not providers:
        local_ws = f"ws://127.0.0.1:{SERVER_PORT}{KATAGO_WS_PATH}"
        local_status = f"http://127.0.0.1:{SERVER_PORT}/status"
        print("Public tunnel disabled by TUNNEL_PROVIDER='none'.")
        print("Local WebSocket URL:", local_ws)
        print("Local Status URL:", local_status)
        return None, local_ws, local_status

    starters = {
        "cloudflare": start_cloudflare_tunnel,
        "pinggy": start_pinggy_tunnel,
        "localhostrun": start_localhostrun_tunnel,
    }
    errors = []
    for provider in providers:
        try:
            return starters[provider]()
        except Exception as exc:
            errors.append(f"{provider}: {exc}")
            print(f"\nTunnel provider failed ({provider}): {exc}")
            print("Trying next tunnel provider..." if provider != providers[-1] else "No tunnel providers left.")

    raise RuntimeError("All tunnel providers failed:\n" + "\n".join(errors))


# ============================================================
# Run everything
# ============================================================

print("Selected mode:", CONFIG_DATA["mode_name"])

if STOP_OLD_PROCESSES:
    stop_old_processes()
else:
    print("\n[1/12] Skipping old process cleanup because STOP_OLD_PROCESSES=False")

print("\n[2/12] Installing notebook dependencies...")
install_dependencies(include_analysis=True)
restore_persistent_cache()
print_system_info()
katago_exec = download_katago()
download_model()
sync_persistent_cache()
write_analysis_config(katago_exec)
sync_persistent_cache()
write_ws_server(katago_exec)

ws_proc = start_ws_server()
test_local_http()
test_local_websocket()

tunnel_providers = selected_tunnel_providers()
if "cloudflare" in tunnel_providers:
    download_cloudflared()
    sync_persistent_cache()
else:
    print("\n[11/12] Skipping cloudflared download because Cloudflare is not selected.")

tunnel_proc, final_wss_url, final_status_url = start_public_tunnel()

print("\nSETUP COMPLETE")
print("\nFinal SWHub WebSocket URL:")
print(final_wss_url)
print("\nStatus URL:")
print(final_status_url)


In [ ]:
%%swhub_action stop
# @title Register stop (runs only from the panel)
# @title Stop KataGo Remote Engine
# Stop the server before closing

import os, signal, subprocess, time

patterns = [
    "katago_ws_server.py",
    "katago analysis",
    "cloudflared tunnel",
    "free.pinggy.io",
    "localhost.run",
]

for line in subprocess.check_output(["ps", "-eo", "pid=,args="], text=True).splitlines():
    line = line.strip()
    if not line:
        continue

    parts = line.split(None, 1)
    if len(parts) < 2:
        continue

    pid = int(parts[0])
    cmd = parts[1]

    if any(p in cmd for p in patterns):
        print("Stopping:", pid, cmd)
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

time.sleep(2)

sync_cache = globals().get("sync_persistent_cache")
if callable(sync_cache):
    sync_cache()

print("Done.")



## Troubleshooting

- **No NVIDIA GPU:** select a GPU in Runtime settings and click Start server again.
- **Connection stops working:** the runtime or temporary connection may have ended. Rerun Start and copy the newly printed address into SWHub.
- **Files download again:** sign in to the same Google Drive account. Only completed, verified files are reused.
- **Package installation fails:** read the first error above the traceback, then click Start server again. If the same error continues, use **Runtime > Disconnect and delete runtime**, reconnect and reopen the panel and click Start server again.
- **A Colab update breaks setup:** open **Runtime > Change runtime type > Runtime Version** and try an available earlier version, such as `2026.07`, as a temporary workaround. Recheck the GPU selection, then click Start server again. Google keeps older versions for a limited time. [Runtime version help](https://research.google.com/colaboratory/runtime-version-faq.html)
- **CUDA libraries are missing:** setup normally repairs these automatically. The first repair downloads about 1.4 GB and can take several minutes. It keeps Colab's PyTorch installation intact.

